In [4]:
#Upload data from google drive
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
drive_path = '/content/drive/My Drive/Programacion/Actuary/freMTPL2/'
df_freq = pd.read_csv(drive_path + 'freMTPL2freq.csv')  #Frequency
df_sev = pd.read_csv(drive_path + 'freMTPL2sev.csv')    #Severity

Mounted at /content/drive


In [5]:
# ==============================================================================
# BLOQUE 1: AGRUPACIÓN Y UNIFICACIÓN DE DATASETS (MERGE)
# ==============================================================================
import pandas as pd

print("--> Iniciando la unificación de bases de datos...")

# 1. Agrupar la base de severidad por ID de póliza
# Sumamos los montos de reclamos por si un cliente tuvo múltiples siniestros
df_sev_grouped = df_sev.groupby('IDpol').agg(
    TotalClaimAmount=('ClaimAmount', 'sum')
).reset_index()

print(f"Base de severidad agrupada por IDpol: {df_sev_grouped.shape[0]} pólizas con siniestros.")

# 2. Unificar con la base de frecuencia (Fila por póliza / Exposición)
# Usamos 'left' para mantener absolutamente todas las pólizas emitidas
df_merged = pd.merge(df_freq, df_sev_grouped, on='IDpol', how='left')

# 3. Tratamiento de valores nulos
# Los clientes que no chocaron van a tener NaN en el monto. Los pasamos a 0.
df_merged['TotalClaimAmount'] = df_merged['TotalClaimAmount'].fillna(0)

# 4. Verificación de consistencia del Merge
print("-" * 60)
print(f"Dimensiones finales del dataset unificado: {df_merged.shape[0]} filas y {df_merged.shape[1]} columnas.")
print(f"Cantidad total de siniestros registrados (ClaimNb): {df_merged['ClaimNb'].sum()}")
print(f"Pólizas con monto mayor a cero: {df_merged[df_merged['TotalClaimAmount'] > 0].shape[0]}")
print("-" * 60)

# Mostrar las primeras filas para validar visualmente
df_merged[['IDpol', 'ClaimNb', 'TotalClaimAmount']].head(10)

--> Iniciando la unificación de bases de datos...
Base de severidad agrupada por IDpol: 24950 pólizas con siniestros.
------------------------------------------------------------
Dimensiones finales del dataset unificado: 678013 filas y 13 columnas.
Cantidad total de siniestros registrados (ClaimNb): 36102
Pólizas con monto mayor a cero: 24944
------------------------------------------------------------


,IDpol,ClaimNb,TotalClaimAmount
0,1.0,1,0.0
1,3.0,1,0.0
2,5.0,1,0.0
3,10.0,1,0.0
4,11.0,1,0.0
5,13.0,1,0.0
6,15.0,1,0.0
7,17.0,1,0.0
8,18.0,1,0.0
9,21.0,1,0.0


In [7]:
# ==============================================================================
# BLOQUE 2: FILTROS ACTUARIALES Y CONTROL DE EXPOSICIÓN
# ==============================================================================
print("--> Aplicando filtros actuariales de control...")

# Guardamos una copia para no alterar el DataFrame anterior
df_filtered = df_merged.copy()

# 1. Filtro de Exposición válida (Mayor a 2 días y menor o igual a 1 año)
df_filtered = df_filtered[(df_filtered['Exposure'] > 0.005) & (df_filtered['Exposure'] <= 1.0)]

# 2. Filtro de Consistencia Lógica: Si hay siniestros, debe haber monto, y si hay monto, debe haber siniestros
# Nota: Esto saca las pólizas con siniestros que costaron €0 o registros huérfanos.
inconsistent_mask = ((df_filtered['ClaimNb'] > 0) & (df_filtered['TotalClaimAmount'] == 0)) | \
                    ((df_filtered['ClaimNb'] == 0) & (df_filtered['TotalClaimAmount'] > 0))

df_filtered = df_filtered[~inconsistent_mask]

# 3. Métricas de control en pantalla
rows_removed = df_merged.shape[0] - df_filtered.shape[0]
print("-" * 60)
print(f"Filas eliminadas por filtros de exposición e inconsistencias: {rows_removed}")
print(f"Dimensiones del dataset filtrado: {df_filtered.shape[0]} filas.")
print(f"Exposición promedio retenida: {df_filtered['Exposure'].mean():.4f} años.")
print(f"Nueva cantidad total de siniestros: {df_filtered['ClaimNb'].sum()}")
print("-" * 60)

# Verificamos los extremos de la exposición para validar el filtro
df_filtered['Exposure'].describe()

--> Aplicando filtros actuariales de control...
------------------------------------------------------------
Filas eliminadas por filtros de exposición e inconsistencias: 13401
Dimensiones del dataset filtrado: 664612 filas.
Exposición promedio retenida: 0.5304 años.
Nueva cantidad total de siniestros: 26388
------------------------------------------------------------


,Exposure
count,664612.000000
mean,0.530418
std,0.362737
min,0.005464
25%,0.180000
50%,0.490000
75%,0.990000
max,1.000000


In [9]:
df_merged['VehPower'].describe()

,VehPower
count,678013.000000
mean,6.454631
std,2.050906
min,4.000000
25%,5.000000
50%,6.000000
75%,7.000000
max,15.000000


In [10]:
# ==============================================================================
# BLOQUE 3: INGENIERÍA DE VARIABLES (FEATURE ENGINEERING) Y CAPPINGS
# ==============================================================================
import numpy as np
import pandas as pd

print("--> Aplicando ingeniería de variables al dataset filtrado...")

# Trabajamos sobre una copia del dataset limpio del bloque anterior
df_features = df_filtered.copy()

# 1. Transformación binaria de combustible (Evita la trampa de la dummy en el GLM)
df_features['Is_Diesel'] = (df_features['VehGas'] == 'Diesel').astype(int)

# 2. Suavizado logarítmico de la densidad de población
df_features['LogDensity'] = np.log(df_features['Density'])

# 3. Truncamiento (Capping) de la cola de alto riesgo de BonusMalus
df_features['BonusMalus_Grouped'] = np.where(df_features['BonusMalus'] >= 130, 130, df_features['BonusMalus'])

# 4. Truncamiento (Capping) de potencias vehiculares extremas o raras
df_features['VehPower_Capped'] = np.where(df_features['VehPower'] >= 9, 9, df_features['VehPower'])

# 5. Binning Actuarial: Rangos de Edad del Conductor (Captura riesgo no lineal en jóvenes)
df_features['DrivAge_Group'] = pd.cut(
    df_features['DrivAge'],
    bins=[17, 24, 34, 44, 54, 64, 110],
    labels=['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
)

# 6. Binning Actuarial: Rangos de Edad del Vehículo
df_features['VehAge_Group'] = pd.cut(
    df_features['VehAge'],
    bins=[-1, 0, 4, 9, 110],
    labels=['0', '1-4', '5-9', '10+']
)

print("-" * 60)
print("¡Variables numéricas, cappings y rangos generados con éxito!")
print("-" * 60)

# Verificación estadística y visual de los rangos generados
print("\nDistribución de grupos de edad del conductor (DrivAge_Group):")
print(df_features['DrivAge_Group'].value_counts().sort_index())

print("\nDistribución de grupos de edad del vehículo (VehAge_Group):")
print(df_features['VehAge_Group'].value_counts().sort_index())

print("-" * 60)
# Mostrar un adelanto de cómo se transformaron los datos
df_features[['DrivAge', 'DrivAge_Group', 'VehAge', 'VehAge_Group', 'VehGas', 'Is_Diesel', 'BonusMalus', 'BonusMalus_Grouped']].head(10)

--> Aplicando ingeniería de variables al dataset filtrado...
------------------------------------------------------------
¡Variables numéricas, cappings y rangos generados con éxito!
------------------------------------------------------------

Distribución de grupos de edad del conductor (DrivAge_Group):
DrivAge_Group
18-24     29659
25-34    138610
35-44    167567
45-54    160476
55-64     97076
65+       71224
Name: count, dtype: int64

Distribución de grupos de edad del vehículo (VehAge_Group):
VehAge_Group
0       53279
1-4    220963
5-9    169204
10+    221166
Name: count, dtype: int64
------------------------------------------------------------


,DrivAge,DrivAge_Group,VehAge,VehAge_Group,VehGas,Is_Diesel,BonusMalus,BonusMalus_Grouped
66,61,55-64,1,1-4,Regular,0,50,50
93,50,45-54,5,5-9,Diesel,1,60,60
199,36,35-44,0,0,Regular,0,85,85
205,51,45-54,0,0,Regular,0,100,100
223,45,45-54,0,0,Regular,0,50,50
287,54,45-54,6,5-9,Diesel,1,50,50
295,34,25-34,0,0,Regular,0,64,64
388,44,35-44,0,0,Regular,0,50,50
396,24,18-24,10,10+,Regular,0,105,105
468,60,55-64,0,0,Regular,0,50,50


In [11]:
# ==============================================================================
# BLOQUE 4: CAPPING DE SEVERIDAD EXTREMA Y EXPORTACIÓN FINAL
# ==============================================================================
print("--> Aplicando capping a los montos de siniestros extremos...")

# 1. Calculamos el percentil 99.5% considerando SOLO las pólizas que tuvieron choques (> 0)
cap_severity_value = df_features[df_features['TotalClaimAmount'] > 0]['TotalClaimAmount'].quantile(0.995)

# 2. Aplicamos el truncamiento en una nueva columna
df_features['ClaimAmount_Capped'] = np.where(
    df_features['TotalClaimAmount'] > cap_severity_value,
    cap_severity_value,
    df_features['TotalClaimAmount']
)

print("-" * 60)
print(f"El umbral de corte para el percentil 99.5% es: € {cap_severity_value:.2f}")
print(f"Monto máximo original: € {df_features['TotalClaimAmount'].max():.2f}")
print(f"Nuevo monto máximo acotado: € {df_features['ClaimAmount_Capped'].max():.2f}")
print("-" * 60)

# 3. EXPORTAR EL DATASET LIMPIO A UN ARCHIVO CSV
# Este CSV será la "fuente única de verdad" para los Notebooks 3 (GLM) y 4 (ML)
df_features.to_csv('mtpl_cleaned_portfolio.csv', index=False)
print("--> ¡Dataset exportado con éxito como 'mtpl_cleaned_portfolio.csv'!")

--> Aplicando capping a los montos de siniestros extremos...
------------------------------------------------------------
El umbral de corte para el percentil 99.5% es: € 35658.65
Monto máximo original: € 4075400.56
Nuevo monto máximo acotado: € 35658.65
------------------------------------------------------------
--> ¡Dataset exportado con éxito como 'mtpl_cleaned_portfolio.csv'!
